# Smith-Waterman: C vs CUDA - Week 7 setup
CEPARCO Integrating Project, Group 3

Checks that the Colab environment works for the project: GPU, `nvcc`, a tiny CUDA
compile-and-run test, the repository, and the test/benchmark input files.

**Before running:** Runtime -> Change runtime type -> choose a **GPU** (usually a T4).

## 1. GPU and compiler

In [30]:
import os
os.environ["PATH"] += ":/usr/local/cuda/bin"      # in case nvcc is not already on PATH
!nvidia-smi
!nvcc --version
!gcc --version | head -1

Thu Nov 12 13:03:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. CUDA smoke test
Compiles and runs a 1,000-thread kernel with the same flags the project will use
(`-arch=native` lets `nvcc` detect the GPU; needs CUDA 11.5 or newer - otherwise use e.g. `-arch=sm_75`).
Expected output: `sum=499500 (expected 499500) -> OK`

In [31]:
%%writefile hello.cu
#include <cstdio>
__global__ void fill(int *x, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) x[i] = i;
}
int main() {
    const int n = 1000;
    int *d, h[1000];
    cudaMalloc(&d, n * sizeof(int));
    fill<<<(n + 255) / 256, 256>>>(d, n);
    cudaMemcpy(h, d, sizeof h, cudaMemcpyDeviceToHost);
    long s = 0;
    for (int i = 0; i < n; i++) s += h[i];
    printf("sum=%ld (expected 499500) -> %s\n", s, s == 499500 ? "OK" : "WRONG");
    cudaFree(d);
    return s == 499500 ? 0 : 1;
}

Writing hello.cu


In [32]:
!nvcc -O2 -arch=native hello.cu -o hello && ./hello

sum=499500 (expected 499500) -> OK


## 3. Get the repository
Option A (recommended): clone the team repository - **edit `REPO_URL` first**.
Option B: upload `smith-waterman-week7.zip` (uncomment the lines).

In [33]:
REPO_URL = "https://github.com/Luis-AQR/CEPARCO_GRP3_IntegratingProject.git"
BRANCH   = "build/week7-requirements"

%cd /content
!rm -rf smith-waterman
!git clone --branch $BRANCH $REPO_URL smith-waterman
%cd smith-waterman
!ls

/content
Cloning into 'smith-waterman'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 37 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 31.14 MiB | 31.36 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/smith-waterman
'[CEPARCO] Group3_IntegratingProjectProposal.pdf'   smith-waterman
'CEPARCO Integrating Project Pitch.mp4'		    SW_Week7_Setup.ipynb
 README.md


## 4. Validate the test sequences
Runs an independent pure-Python Smith-Waterman on the four small test inputs and compares
with the expected scores, coordinates, alignments and (for the worked example) the full
score matrix; also checks the format of the benchmark inputs.

In [34]:
%cd /content/smith-waterman/smith-waterman
!ls
!python3 scripts/check_expected.py

/content/smith-waterman/smith-waterman
colab  data  README.md	scripts
Test sequences (pure-Python reference, match +1 / mismatch -1 / gap -2):
  ok   test_paper.fasta         4 x 5   score=3   end=(4, 5) GCT
  ok   test_identical.fasta     8 x 8   score=8   end=(8, 8) ACGTACGT
  ok   test_nomatch.fasta       8 x 8   score=0   end=(0, 0) (no alignment)
  ok   test_gap.fasta          12 x 13  score=10  end=(12, 13) ACGTCA-TGCATG

Benchmark sequences (format check only; scored by the C baseline in Week 9):
  ok   bench_1000.fasta       1000 x 1000, alphabet ACGT
  ok   bench_5000.fasta       5000 x 5000, alphabet ACGT
  ok   bench_10000.fasta      10000 x 10000, alphabet ACGT

All checks passed.


## 5. Benchmark inputs are reproducible
Regenerates each benchmark file from its seed and compares it with the committed copy.
All three lines should say `identical`.

In [35]:
%%bash
for n in 1000 5000 10000; do
  python3 scripts/gen_sequences.py --length $n --seed 1 --out /tmp/regen_$n.fasta > /dev/null
  if cmp -s /tmp/regen_$n.fasta data/bench_$n.fasta; then
    echo "bench_$n: identical"
  else
    echo "bench_$n: DIFFERENT (committed file is authoritative; tell the team)"
  fi
done

bench_1000: identical
bench_5000: identical
bench_10000: identical


## 6. Record the environment
Colab's GPU depends on availability. Paste this line into the README / report so every
timing can be traced to the machine that produced it.

In [36]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
!nvcc --version | tail -1
!python3 --version

Tesla T4, 580.82.07, 15360 MiB
Build cuda_13.0.r13.0/compiler.36424714_0
Python 3.13.16
